# Diarization

In [1]:
from pathlib import Path
import sys
import os
import torch

from pyannote.audio import Pipeline
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.diarization.pyannote import run_diarization_pipeline

W0617 14:47:27.556000 15480 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
e:\Project\daic-woz-audio-analysis\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. T

In [2]:
DIARIZATION_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "diarization"

LOG_DIR = PROJECT_ROOT / "logs"
SUCCESS_LOG = LOG_DIR / "diarization_success.csv"
FAILED_LOG = LOG_DIR / "diarization_failures.csv"

RAW_DIR = PROJECT_ROOT / "data" / "raw"
AUDIO_DIR = PROJECT_ROOT / "data" / "audio"
TRANSCRIPT_DIR = PROJECT_ROOT / "data" / "transcript"

for directory in [
    RAW_DIR,
    AUDIO_DIR,
    TRANSCRIPT_DIR,
    DIARIZATION_OUTPUT_DIR,
    LOG_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

audio_files = sorted(AUDIO_DIR.glob("*_AUDIO.wav"))

print(f"Found {len(audio_files)} audio files")

# %%
load_dotenv(PROJECT_ROOT / ".env")



Found 275 audio files


True

In [3]:
HF_TOKEN = os.getenv("HF_TOKEN")
assert HF_TOKEN is not None, "HF_TOKEN not found in .env"

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN,
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

pipeline.to(device)

print(f"Device: {device}")

Device: cuda


In [ ]:
run_diarization_pipeline(
    pipeline=pipeline,
    audio_paths=audio_files,
    output_dir=DIARIZATION_OUTPUT_DIR,
    success_log=SUCCESS_LOG,
    failure_log=FAILED_LOG,
    overwrite=False,
)

Running Diarization:   0%|          | 0/275 [00:00<?, ?audio/s]e:\Project\daic-woz-audio-analysis\.venv\Lib\site-packages\pyannote\audio\utils\reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
e:\Project\daic-woz-audio-analysis\.venv\Lib\site-packages\pyannote\audio\models\blocks\pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  std = sequences.std(dim=-1, correction=1)
Running Diarization:   0%|          | 1/275 [00:14<

[SUCCESS] 300


Running Diarization:   1%|          | 2/275 [00:32<1:14:58, 16.48s/audio]

[SUCCESS] 301


Running Diarization:   1%|          | 3/275 [00:49<1:14:56, 16.53s/audio]

[SUCCESS] 302
